In [ ]:
import numpy as np

import mne

In [ ]:
!pwd

In [ ]:
sample_data_folder = mne.datasets.sample.data_path(path= "/Users/uarc/Downloads")
sample_data_raw_file = (
    sample_data_folder / "MEG" / "sample" / "sample_audvis_filt-0-40_raw.fif"
)
raw = mne.io.read_raw_fif(sample_data_raw_file, verbose=True)

In [ ]:
import matplotlib.pyplot as plt


raw.compute_psd(fmax=50).plot(picks="data", exclude="bads", amplitude=False)
raw.plot(duration=5, n_channels=30)

plt.savefig("EEG_plt.png")

plt.show()


In [ ]:
# set up and fit the ICA
ica = mne.preprocessing.ICA(n_components=20, random_state=97, max_iter=800)
ica.fit(raw)
ica.exclude = [1, 2]  # details on how we picked these are omitted here
ica.plot_properties(raw, picks=ica.exclude)
plt.savefig("EEGplot1.png")

plt.show()


In [ ]:
ica.plot_properties(raw, picks=[3, 4])

In [ ]:
orig_raw = raw.copy()
raw.load_data()
ica.apply(raw)

# show some frontal channels to clearly illustrate the artifact removal
chs = [
    "MEG 0111",
    "MEG 0121",
    "MEG 0131",
    "MEG 0211",
    "MEG 0221",
    "MEG 0231",
    "MEG 0311",
    "MEG 0321",
    "MEG 0331",
    "MEG 1511",
    "MEG 1521",
    "MEG 1531",
    "EEG 001",
    "EEG 002",
    "EEG 003",
    "EEG 004",
    "EEG 005",
    "EEG 006",
    "EEG 007",
    "EEG 008",
]
chan_idxs = [raw.ch_names.index(ch) for ch in chs]
orig_raw.plot(order=chan_idxs, start=12, duration=4)
raw.plot(order=chan_idxs, start=12, duration=4)

In [ ]:
events = mne.find_events(raw, stim_channel="STI 014")
print(events[:5])  # show the first 5

In [ ]:
reject_criteria = dict(
    mag=4000e-15,  # 4000 fT
    grad=4000e-13,  # 4000 fT/cm
    eeg=150e-6,  # 150 µV
    eog=250e-6,
)  # 250 µV

In [ ]:
import ipyevents
import nibabel


In [ ]:
# Example EEG signal (10 minutes at 500 Hz)
eeg_signal = np.random.randn(300000)  # 500 samples/second * 600 seconds (10 minutes)

# Set window size to 1 second (500 samples)
window_size = 500

# Segment the EEG data into 1-second windows
segments = [eeg_signal[i:i + window_size] for i in range(0, len(eeg_signal), window_size)]


In [ ]:
import numpy as np

def extract_features(segments):
    features = []
    for segment in segments:
        feature_vector = [
            np.mean(segment),
            np.std(segment),
            np.min(segment),
            np.max(segment),
            np.median(segment)
        ]
        features.append(feature_vector)
    return np.array(features)

features = extract_features(segments)


In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# Simulated segment data for example purposes
segments = np.random.randn(600, 500)  # 600 segments, 500 timesteps per segment (1-second intervals, for example)

# Define the dimensions based on your data
num_segments = len(segments)  # Number of segments (600 in this case)
timesteps = segments.shape[1]  # Number of timesteps per segment (500 in this case)
num_features = 1  # Assuming univariate data (like a single EEG channel)

# Reshape segments to match the required input shape for a transformer model
reshaped_segments = segments.reshape((num_segments, timesteps, num_features))

# If you're normalizing the data (optional)
scaler = StandardScaler()
normalized_segments = scaler.fit_transform(reshaped_segments.reshape(-1, num_features))  # Flatten, scale, and reshape back
normalized_segments = normalized_segments.reshape(num_segments, timesteps, num_features)

print(f"Reshaped Segments Shape: {reshaped_segments.shape}")


In [ ]:
import torch
import torch.nn as nn

class TimeSeriesTransformer(nn.Module):
    def __init__(self, input_dim, num_heads, num_classes, num_encoder_layers=3, dropout=0.1):
        super(TimeSeriesTransformer, self).__init__()
        self.positional_encoding = nn.Embedding(input_dim, input_dim)  # Positional encoding
        # Set batch_first=True
        encoder_layer = nn.TransformerEncoderLayer(d_model=input_dim, nhead=num_heads, dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)
        self.fc = nn.Linear(input_dim, num_classes)  # Final classifier layer for classification tasks

    def forward(self, x):
        # Apply positional encoding
        x = self.positional_encoding(x)
        # Pass through transformer encoder
        x = self.transformer_encoder(x)
        # Mean pooling across the time dimension
        x = x.mean(dim=1)
        # Final classification layer
        x = self.fc(x)
        return x

# Example model initialization
input_dim = 8  # Make sure input_dim is divisible by num_heads
num_heads = 4  # Number of attention heads in multi-head attention
num_classes = 2  # Example: binary classification
model = TimeSeriesTransformer(input_dim, num_heads, num_classes)

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

